# MedGemma — Train/val/test split (Colab, CPU runtime)

Reads the prepared reports + image paths and builds the dataset:

1. records (text + image path) for DMID and VinDr,
2. BI-RADS strata,
3. **70/15/15** stratified split,
4. saves a `DatasetDict` to Drive for the training notebook to load.

Run `medgemma-image-processing.ipynb` and `medgemma-report-processing.ipynb`
beforehand to fill the `images-processed/` and `reports-processed/` (DMID) and
`reports/` (VinDr) folders.

In [ ]:
import glob
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from datasets import ClassLabel, Dataset, DatasetDict

SEED = 2

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
ROOT_DIR = Path("/content/drive/MyDrive/MedGemma2026/main")
DATA_DIR = ROOT_DIR / "data"
SPLIT_DIR = DATA_DIR / "split"

# DMID
DMID_TIF_DIR = DATA_DIR / "dmid" / "images-original"
DMID_PNG_DIR = DATA_DIR / "dmid" / "images-processed"

# VinDr-Mammo
VINDR_BASE = DATA_DIR / "vindr-mammo"
VINDR_REPORTS = VINDR_BASE / "reports"
VINDR_ANN = VINDR_BASE / "breast-level_annotations.csv"
VINDR_PNG_DIR = VINDR_BASE / "images-processed"

SOURCES = [
    {
        "name": "dmid",
        "report_path": DATA_DIR / "dmid" / "reports-processed",
        "image_path": DMID_PNG_DIR,
        "image_ext": ".png",
        "image_case": "upper",
    },
    {
        "name": "vindr-mammo",
        "report_path": VINDR_REPORTS,
        "image_root": VINDR_PNG_DIR,
        "annotation_csv": VINDR_ANN,
        "image_ext": ".png",
        "image_case": "lower",
        "nested_by_study": True,
    },
]

for s in SOURCES:
    print(
        f"Source {s['name']!r}: reports={s['report_path']}, "
        f"images={s.get('image_path') or s.get('image_root')}"
    )

Source 'dmid': reports=/content/drive/MyDrive/MedGemma2026/main/data/dmid/reports-processed, images=/content/drive/MyDrive/MedGemma2026/main/data/dmid/images-processed
Source 'vindr-mammo': reports=/content/drive/MyDrive/MedGemma2026/main/data/vindr-mammo/reports, images=/content/drive/MyDrive/MedGemma2026/main/data/vindr-mammo/images-processed


In [4]:
def load_data(source: dict) -> list[dict]:
    """Load (text, image-path) pairs from one source, tagging each record with
    `source` so downstream code can slice by origin."""
    report_path = str(source["report_path"])
    image_ext = source.get("image_ext", ".tif")
    image_case = source.get("image_case", "upper")
    nested = bool(source.get("nested_by_study"))

    study_of: dict[str, str] = {}
    if nested:
        ann = pd.read_csv(source["annotation_csv"])
        study_of = dict(zip(ann["image_id"].astype(str), ann["study_id"].astype(str)))
        image_root = Path(source["image_root"])
    else:
        image_path = Path(source["image_path"])

    data = []
    missing_study = 0
    for text_file_path in glob.glob(os.path.join(report_path, "*.txt")):
        base_name = os.path.splitext(os.path.basename(text_file_path))[0]
        image_base = base_name.upper() if image_case == "upper" else base_name.lower()

        if nested:
            study_id = study_of.get(base_name) or study_of.get(image_base)
            if study_id is None:
                missing_study += 1
                continue
            image_file = str(image_root / study_id / f"{base_name}{image_ext}")
        else:
            image_file = os.path.join(str(image_path), f"{image_base}{image_ext}")

        with open(text_file_path) as stream:
            raw_content = stream.read()

        data.append(
            {
                "id": base_name,
                "source": source["name"],
                "text": raw_content.strip(),
                "image": image_file,
            }
        )

    if nested and missing_study:
        print(
            f"  [{source['name']}] {missing_study} report(s) had no study_id in the "
            f"annotation CSV and were skipped"
        )
    return data


raw_data: list[dict] = []
for s in SOURCES:
    if not s["report_path"].exists():
        print(f"Skipping source {s['name']!r}: {s['report_path']} not found")
        continue
    chunk = load_data(s)
    print(f"Loaded {len(chunk):>5d} records from source {s['name']!r}")
    raw_data.extend(chunk)

print(
    f"Total: {len(raw_data)} records across "
    f"{len({r['source'] for r in raw_data})} source(s)"
)

raw_data[:1]

Loaded   508 records from source 'dmid'
Loaded 20000 records from source 'vindr-mammo'
Total: 20508 records across 2 source(s)


[{'id': 'Img001',
  'source': 'dmid',
  'text': 'Breast Composition: There are scattered areas of fibroglandular density (ACR B).\n\nBI-RADS: 3, 5\n\nFindings:\n- Irregular ill-defined soft opacity with microcalcifications suggests malignant lesion (BI-RADS 5).\n- Small well defined soft nodular opacity -- benign lesion (BI-RADS 3).\n- Benign vascular calcification seen.\n- Skin and nipple appear normal.\n- No axillary adenopathy.',
  'image': '/content/drive/MyDrive/MedGemma2026/main/data/dmid/images-processed/IMG001.png'}]

In [5]:
def extract_birads_level(text: str) -> str:
    match = re.search(r"BI[-\s]?RADS:\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    return (
        re.sub(r"\s*,\s*", " and ", match.group(1).strip().lower())
        if match
        else "unknown"
    )


def extract_acr_density(text: str) -> str | None:
    match = re.search(r"\(ACR[\s-]+([A-D])\s*\)", text, re.IGNORECASE)
    return match.group(1).upper() if match else None


_BIRADS_DIGIT_RE = re.compile(r"([0-6])([abc])?", re.IGNORECASE)


def birads_strata(birads: str) -> str:
    matches = _BIRADS_DIGIT_RE.findall(birads or "")
    if not matches:
        return birads or "unknown"
    best = max(matches, key=lambda m: (int(m[0]), m[1].lower() or ""))
    return f"{best[0]}{best[1].lower()}"


for record in raw_data:
    record["birads"] = extract_birads_level(record["text"])
    record["birads_strata"] = birads_strata(record["birads"])
    record["acr_density"] = extract_acr_density(record["text"])
    if record["birads"] == "unknown":
        print(f"Unknown BIRADS in {record['id']} (source={record['source']})")

# Drop records whose BI-RADS stratum is 0 or 6 -- the prompts no longer list
# those categories, so we shouldn't be training on them either.
EXCLUDED_BIRADS = {"0", "6"}

_dropped = [r for r in raw_data if r["birads_strata"] in EXCLUDED_BIRADS]
raw_data = [r for r in raw_data if r["birads_strata"] not in EXCLUDED_BIRADS]
if _dropped:
    print(
        f"Dropped {len(_dropped)} record(s) with BI-RADS in {sorted(EXCLUDED_BIRADS)}:"
    )
    for r in _dropped[:10]:
        print(f"  - id={r['id']} source={r['source']} birads={r['birads']!r}")
    if len(_dropped) > 10:
        print(f"  ... and {len(_dropped) - 10} more")

birads_distribution = Counter(r["birads_strata"] for r in raw_data)
print(f"BIRADS strata distribution ({len(raw_data)} records):")
for level, count in sorted(birads_distribution.items(), key=lambda x: (-x[1], x[0])):
    print(f"- BIRADS {level:<6}: {count:5d} ({count / len(raw_data) * 100:.2f}%)")

Dropped 1 record(s) with BI-RADS in ['0', '6']:
  - id=Img106 source=dmid birads='0'
BIRADS strata distribution (20507 records):
- BIRADS 1     : 13615 (66.39%)
- BIRADS 2     :  4701 (22.92%)
- BIRADS 3     :  1051 (5.13%)
- BIRADS 4     :   765 (3.73%)
- BIRADS 5     :   246 (1.20%)
- BIRADS 4c    :    54 (0.26%)
- BIRADS 4a    :    39 (0.19%)
- BIRADS 4b    :    36 (0.18%)


In [6]:
# Optional smoke-test subsampling
SAMPLE_PER_SOURCE: int | None = None

if SAMPLE_PER_SOURCE:
    _rng = random.Random(SEED)
    by_source: dict[str, list[dict]] = defaultdict(list)
    for r in raw_data:
        by_source[r["source"]].append(r)

    sampled: list[dict] = []
    for src, records in by_source.items():
        by_strata: dict[str, list[dict]] = defaultdict(list)
        for r in records:
            by_strata[r["birads_strata"]].append(r)
        for bucket in by_strata.values():
            _rng.shuffle(bucket)
        picked: list[dict] = []
        strata_keys = sorted(by_strata)
        while len(picked) < SAMPLE_PER_SOURCE and any(
            by_strata[k] for k in strata_keys
        ):
            for k in strata_keys:
                if by_strata[k] and len(picked) < SAMPLE_PER_SOURCE:
                    picked.append(by_strata[k].pop())
        sampled.extend(picked)
        print(f"[{src}] sampled {len(picked)} of {len(records)} records")

    raw_data = sampled
    print(f"\nAfter subsampling: {len(raw_data)} total records")
else:
    print("SAMPLE_PER_SOURCE is None -- using full dataset.")

SAMPLE_PER_SOURCE is None -- using full dataset.


In [7]:
# --- Split: minority strata (everything except 1 & 2) keep a 70/15/15 split;
#     BI-RADS 1 & 2 then fill val/test up to a fixed size, split evenly.
VAL_SIZE = 1000
TEST_SIZE = 1000
VAL_FRAC = 0.15
TEST_FRAC = 0.15
MIN_PER_STRATUM_TO_SPLIT = 6   # strata smaller than this go entirely to train
FILL_STRATA = ["1", "2"]       # these fill the remainder of the val/test budget


def _cap_keep_dmid(items, cap, rng):
    """Downsample `items` to `cap`, keeping every dmid record and dropping only
    vindr-mammo records to make room. If dmid alone already meets the cap, all
    dmid are still kept (full DMID is required)."""
    dmid = [r for r in items if r["source"] == "dmid"]
    vindr = [r for r in items if r["source"] != "dmid"]
    if len(dmid) >= cap:
        return dmid
    rng.shuffle(vindr)
    return dmid + vindr[: cap - len(dmid)]


def _fill_eval(items, target_val, target_test, val_frac, test_frac, rng):
    """Split this stratum's DMID across train/val/test BY RATIO, then fill the
    val/test targets up to size with vindr-mammo. Returns (val, test, train);
    nothing is dropped (every dmid record is used)."""
    dmid = [r for r in items if r["source"] == "dmid"]
    vindr = [r for r in items if r["source"] != "dmid"]
    rng.shuffle(dmid)
    rng.shuffle(vindr)
    nd = len(dmid)
    nd_test = round(nd * test_frac)
    nd_val = round(nd * val_frac)
    d_test = dmid[:nd_test]
    d_val = dmid[nd_test : nd_test + nd_val]
    d_train = dmid[nd_test + nd_val :]
    v_val = max(0, target_val - len(d_val))
    v_test = max(0, target_test - len(d_test))
    val = d_val + vindr[:v_val]
    test = d_test + vindr[v_val : v_val + v_test]
    train = d_train + vindr[v_val + v_test :]
    return val, test, train


_rng = random.Random(SEED)
by_strata: dict[str, list[dict]] = defaultdict(list)
for r in raw_data:
    by_strata[r["birads_strata"]].append(r)

train_records, val_records, test_records = [], [], []

# 1) Minority strata: ordinary 70/15/15 stratified split.
others_val = others_test = 0
for stratum, items in by_strata.items():
    if stratum in FILL_STRATA:
        continue
    items = items[:]
    _rng.shuffle(items)
    n = len(items)
    if n < MIN_PER_STRATUM_TO_SPLIT:
        train_records.extend(items)
        continue
    n_test = max(1, round(n * TEST_FRAC))
    n_val = max(1, round(n * VAL_FRAC))
    test_records.extend(items[:n_test])
    val_records.extend(items[n_test : n_test + n_val])
    train_records.extend(items[n_test + n_val :])
    others_val += n_val
    others_test += n_test

# 2) BI-RADS 1 & 2: place their DMID by ratio, then fill val/test up to
#    (VAL_SIZE, TEST_SIZE) with vindr; the budget is split evenly 1 & 2.
val_budget = max(0, VAL_SIZE - others_val)
test_budget = max(0, TEST_SIZE - others_test)
k = len(FILL_STRATA)
for j, stratum in enumerate(FILL_STRATA):
    n_val = val_budget // k + (val_budget % k if j == 0 else 0)
    n_test = test_budget // k + (test_budget % k if j == 0 else 0)
    v, t, tr = _fill_eval(by_strata.get(stratum, []), n_val, n_test, VAL_FRAC, TEST_FRAC, _rng)
    val_records.extend(v)
    test_records.extend(t)
    train_records.extend(tr)

# Rebalance TRAIN (val/test now also capped on 1 & 2 above)
# Oversampling replicates minority records: the FIRST copy of each case is the
# original (augment=False, kept clean); every extra copy is a duplicate
# (augment=True) that the train notebook augments live, so duplicates vary.
TARGET_TRAIN_COUNTS = {
    "1":  700,     # downsample (shared 700 cap with val/test)
    "2":  700,     # downsample (shared 700 cap with val/test)
    "5":   500,    # oversample (~3x)
    "4a":  150,    # oversample (~5-6x; safe with live augmentation)
    "4b":  150,
    "4c":  200,
    # "3" and "4" are kept as-is (not in this dict).
}

# Original TRAIN composition BEFORE rebalancing (real records, pre cap/oversample).
_orig_train = Counter(r["birads_strata"] for r in train_records)
print()
print("Original TRAIN per BI-RADS (pre-rebalance, before any cap/oversample):")
for level, count in sorted(_orig_train.items(), key=lambda x: (-x[1], x[0])):
    print(f"  BIRADS {level:<6}: {count:5d}")
print(f"  Total: {len(train_records)}")

# Secondary balance: within each BI-RADS stratum, even out ACR density toward
# uniform (A/B/C/D), capped so rare densities are not over-duplicated. BI-RADS is
# PRIMARY (each stratum still hits its target total); ACR is SECONDARY.
ACR_MAX_OVERSAMPLE = 6   # a (stratum, density) cell may be replicated at most ~6x


def _take_keep_dmid(items, target, rng):
    """Resample `items` to exactly `target`, keeping all dmid first. Downsample
    drops only vindr-mammo; oversample replicates (capped by the caller)."""
    if target <= 0 or not items:
        return []
    n = len(items)
    if target <= n:
        kept = _cap_keep_dmid(items, target, rng)            # downsample, keep dmid
        return [{**r, "augment": False} for r in kept]      # all originals (no duplicates)
    base = items[:]
    rng.shuffle(base)
    reps, rem = divmod(target, n)
    # First full copy is the ORIGINAL of each case (kept clean); every extra
    # copy is a duplicate flagged for augmentation in the train notebook.
    out = [{**r, "augment": False} for r in base]
    for _ in range(reps - 1):
        out.extend({**r, "augment": True} for r in base)
    out.extend({**r, "augment": True} for r in base[:rem])
    return out                                               # oversample


def balance_acr_within(items, target, rng, acr_cap=ACR_MAX_OVERSAMPLE):
    """Hit the BI-RADS `target` for this stratum while spreading the count across
    ACR densities as evenly as data + cap allow (round-robin fill). Quota a
    rare/absent density cannot absorb is redistributed to densities with
    headroom, so the stratum total (BI-RADS, primary) is preserved."""
    by_acr: dict = defaultdict(list)
    for r in items:
        by_acr[r["acr_density"]].append(r)
    classes = [a for a in by_acr if by_acr[a]]
    if not classes or target <= 0:
        return _take_keep_dmid(items, target, rng)
    cap = {a: len(by_acr[a]) * acr_cap for a in classes}     # max from each density
    alloc = {a: 0 for a in classes}
    remaining = target
    active = [a for a in classes if alloc[a] < cap[a]]
    while remaining > 0 and active:
        share = max(1, remaining // len(active))
        for a in list(active):
            give = min(share, cap[a] - alloc[a], remaining)
            alloc[a] += give
            remaining -= give
            if alloc[a] >= cap[a]:
                active.remove(a)
            if remaining <= 0:
                break
    out: list[dict] = []
    for a in classes:
        if alloc[a] > 0:
            out.extend(_take_keep_dmid(by_acr[a], alloc[a], rng))
    rng.shuffle(out)
    return out


_rng_rb = random.Random(SEED)
_by_stratum: dict[str, list[dict]] = defaultdict(list)
for r in train_records:
    _by_stratum[r["birads_strata"]].append(r)

_rebalanced: list[dict] = []
for stratum, items in _by_stratum.items():
    # BI-RADS target (None -> keep the stratum's current size); ACR evened within.
    target = TARGET_TRAIN_COUNTS.get(stratum, len(items))
    _rebalanced.extend(balance_acr_within(items, target, _rng_rb))

train_records = _rebalanced
_rng_rb.shuffle(train_records)

print("\nAfter rebalancing TRAIN:")
_c = Counter(r["birads_strata"] for r in train_records)
for level, count in sorted(_c.items(), key=lambda x: (-x[1], x[0])):
    print(f"  BIRADS {level:<6}: {count:5d}")
print(f"  Total: {len(train_records)}")
print("\n  ACR density within each BI-RADS stratum (train, after rebalancing):")
_strata_order = sorted({r["birads_strata"] for r in train_records})
print(f"    {'BIRADS':<8}{'A':>7}{'B':>7}{'C':>7}{'D':>7}{'total':>8}")
_acr_tot = Counter()
for _st in _strata_order:
    _items = [r for r in train_records if r["birads_strata"] == _st]
    _ca = Counter(r["acr_density"] for r in _items)
    for _d in "ABCD":
        _acr_tot[_d] += _ca.get(_d, 0)
    print(f"    {_st:<8}" + "".join(f"{_ca.get(_d, 0):>7}" for _d in "ABCD") + f"{len(_items):>8}")
print(f"    {'ALL':<8}" + "".join(f"{_acr_tot[_d]:>7}" for _d in "ABCD") + f"{sum(_acr_tot.values()):>8}")

birads_counts = Counter(r["birads_strata"] for r in raw_data)
acr_counts = Counter(r["acr_density"] for r in raw_data)
birads_class_label = ClassLabel(names=sorted(birads_counts))
acr_class_label = ClassLabel(names=sorted(k for k in acr_counts if k is not None))


def _to_split(records):
    return (
        Dataset.from_list(records)
        .cast_column("birads_strata", birads_class_label)
        .cast_column("acr_density", acr_class_label)
    )


data = DatasetDict(
    {
        "train": _to_split(train_records),
        "validation": _to_split(val_records),
        "test": _to_split(test_records),
    }
)


def _describe(name, split):
    print(f"\n{name} ({len(split)} records):")
    by_b = Counter(split["birads_strata"])
    by_s = Counter(split["source"])
    for level, count in sorted(by_b.items(), key=lambda x: (-x[1], x[0])):
        label = birads_class_label.int2str(level) if isinstance(level, int) else level
        print(f"  BIRADS {label:<6}: {count:5d} ({count / len(split) * 100:.2f}%)")
    print("  ACR density:")
    by_a = Counter(split["acr_density"])
    for level, count in sorted(by_a.items(), key=lambda x: (-x[1], str(x[0]))):
        label = acr_class_label.int2str(level) if isinstance(level, int) else level
        print(f"    ACR {label:<4}: {count:5d} ({count / len(split) * 100:.2f}%)")
    print("  by source:")
    for src, count in sorted(by_s.items(), key=lambda x: (-x[1], x[0])):
        print(f"    - {src:<12}: {count:5d}")


for _name in ["train", "validation", "test"]:
    _describe(_name, data[_name])
data


Original TRAIN per BI-RADS (pre-rebalance, before any cap/oversample):
  BIRADS 1     : 12943
  BIRADS 2     :  4031
  BIRADS 3     :   735
  BIRADS 4     :   535
  BIRADS 5     :   172
  BIRADS 4c    :    38
  BIRADS 4a    :    27
  BIRADS 4b    :    26
  Total: 18507

After rebalancing TRAIN:
  BIRADS 3     :   735
  BIRADS 1     :   700
  BIRADS 2     :   700
  BIRADS 4     :   535
  BIRADS 5     :   500
  BIRADS 4c    :   200
  BIRADS 4a    :   150
  BIRADS 4b    :   150
  Total: 3670

  ACR density within each BI-RADS stratum (train, after rebalancing):
    BIRADS        A      B      C      D   total
    1           175    175    175    175     700
    2            84    206    205    205     700
    3           102    211    211    211     735
    4            18    172    173    172     535
    4a           18     72     60      0     150
    4b           51     48     51      0     150
    4c           42     86     60     12     200
    5            36    217    217     30  

Casting the dataset:   0%|          | 0/3670 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3670 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]


train (3670 records):
  BIRADS 3     :   735 (20.03%)
  BIRADS 1     :   700 (19.07%)
  BIRADS 2     :   700 (19.07%)
  BIRADS 4     :   535 (14.58%)
  BIRADS 5     :   500 (13.62%)
  BIRADS 4c    :   200 (5.45%)
  BIRADS 4a    :   150 (4.09%)
  BIRADS 4b    :   150 (4.09%)
  ACR density:
    ACR B   :  1187 (32.34%)
    ACR C   :  1152 (31.39%)
    ACR D   :   805 (21.93%)
    ACR A   :   526 (14.33%)
  by source:
    - vindr-mammo :  2642
    - dmid        :  1028

validation (1000 records):
  BIRADS 1     :   336 (33.60%)
  BIRADS 2     :   335 (33.50%)
  BIRADS 3     :   158 (15.80%)
  BIRADS 4     :   115 (11.50%)
  BIRADS 5     :    37 (3.70%)
  BIRADS 4c    :     8 (0.80%)
  BIRADS 4a    :     6 (0.60%)
  BIRADS 4b    :     5 (0.50%)
  ACR density:
    ACR C   :   756 (75.60%)
    ACR B   :   119 (11.90%)
    ACR D   :   116 (11.60%)
    ACR A   :     9 (0.90%)
  by source:
    - vindr-mammo :   923
    - dmid        :    77

test (1000 records):
  BIRADS 1     :   336 (33.60%)

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'text', 'image', 'birads', 'birads_strata', 'acr_density', 'augment'],
        num_rows: 3670
    })
    validation: Dataset({
        features: ['id', 'source', 'text', 'image', 'birads', 'birads_strata', 'acr_density'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['id', 'source', 'text', 'image', 'birads', 'birads_strata', 'acr_density'],
        num_rows: 1000
    })
})

In [8]:
# Three-stage size summary
# (1) START      : records available in the full merged corpus (pre-split)
# (2) CHOSEN     : how many we allocated to train / val / test
#                  (train_chosen = pre-rebalance pool, before cap/oversample)
# (3) AFTER AUG  : final train size after rebalancing
#                  (cap BI-RADS 1 & 2 down; oversample 5/4a/4b/4c up)
# delta = train_aug - train_chosen  (positive = oversampled, negative = capped)
STRATA_ORDER = ["1", "2", "3", "4", "4a", "4b", "4c", "5"]
ACR_ORDER = ["A", "B", "C", "D"]

start_b = Counter(r["birads_strata"] for r in raw_data)
val_b = Counter(r["birads_strata"] for r in val_records)
test_b = Counter(r["birads_strata"] for r in test_records)
final_b = Counter(r["birads_strata"] for r in train_records)   # after augmentation
# _orig_train was captured just before rebalancing (= train_chosen)

print("Per BI-RADS:  (1) start  ->  (2) chosen [train/val/test]  ->  (3) train after-aug")
hdr = (f"{'birads':<7}{'start':>7}{'train_chosen':>14}{'val':>6}{'test':>6}"
       f"{'train_aug':>11}{'delta':>8}")
print(hdr)
print("-" * len(hdr))
for k in STRATA_ORDER:
    s = start_b.get(k, 0)
    tc = _orig_train.get(k, 0)
    v = val_b.get(k, 0)
    t = test_b.get(k, 0)
    fa = final_b.get(k, 0)
    print(f"{k:<7}{s:>7}{tc:>14}{v:>6}{t:>6}{fa:>11}{fa - tc:>+8}")
tot_s = sum(start_b.values())
tot_tc = sum(_orig_train.values())
print(f"{'TOTAL':<7}{tot_s:>7}{tot_tc:>14}{sum(val_b.values()):>6}"
      f"{sum(test_b.values()):>6}{len(train_records):>11}{len(train_records) - tot_tc:>+8}")

# START distribution on the other axes (full corpus, pre-split)
start_acr = Counter(r["acr_density"] for r in raw_data)
start_src = Counter(r["source"] for r in raw_data)
print()
print("START corpus by ACR density:")
for k in ACR_ORDER:
    print(f"  ACR {k}: {start_acr.get(k, 0):>6}")
print(f"  (none/unparsed): {start_acr.get(None, 0):>6}")
print()
print("START corpus by source:")
for k, n in sorted(start_src.items(), key=lambda x: -x[1]):
    print(f"  {k:<12}: {n:>6}")
print()
print(f"START total records: {len(raw_data)}")


Per BI-RADS:  (1) start  ->  (2) chosen [train/val/test]  ->  (3) train after-aug
birads   start  train_chosen   val  test  train_aug   delta
-----------------------------------------------------------
1        13615         12943   336   336        700  -12243
2         4701          4031   335   335        700   -3331
3         1051           735   158   158        735      +0
4          765           535   115   115        535      +0
4a          39            27     6     6        150    +123
4b          36            26     5     5        150    +124
4c          54            38     8     8        200    +162
5          246           172    37    37        500    +328
TOTAL    20507         18507  1000  1000       3670  -14837

START corpus by ACR density:
  ACR A:    179
  ACR B:   2111
  ACR C:  15477
  ACR D:   2740
  (none/unparsed):      0

START corpus by source:
  vindr-mammo :  20000
  dmid        :    507

START total records: 20507


In [9]:
SPLIT_DIR.parent.mkdir(parents=True, exist_ok=True)
data.save_to_disk(str(SPLIT_DIR))
print(f"Saved prepared DatasetDict to {SPLIT_DIR}")
print({k: len(v) for k, v in data.items()})

Saving the dataset (0/1 shards):   0%|          | 0/3670 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved prepared DatasetDict to /content/drive/MyDrive/MedGemma2026/main/data/split
{'train': 3670, 'validation': 1000, 'test': 1000}
